In [ ]:
!pip install -q datasets pandas

In [ ]:
# ============================================
# STEP 1: DOWNLOAD ELECTRONICS DATASET FILES
# ============================================

!pip install -q huggingface_hub

from huggingface_hub import snapshot_download
import glob
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

# Mount Google Drive for saving results
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
OUTPUT_DIR = '/content/drive/MyDrive/amazon_audio_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Downloading Electronics dataset files...")
print("This will download the raw JSONL files directly\n")

# Download Electronics files - both reviews and metadata
download_dir = snapshot_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset",
    allow_patterns=["*Electronics*"]  # Get all Electronics files
)

# Find the downloaded files
review_files = glob.glob(os.path.join(download_dir, "**", "*review*Electronics*"), recursive=True)
meta_files = glob.glob(os.path.join(download_dir, "**", "*meta*Electronics*"), recursive=True)

print(f"Download directory: {download_dir}")
print(f"\nReview files found: {len(review_files)}")
for f in review_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  - {os.path.basename(f)} ({size_mb:.1f} MB)")

print(f"\nMetadata files found: {len(meta_files)}")
for f in meta_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  - {os.path.basename(f)} ({size_mb:.1f} MB)")

# Check if files are gzipped
is_gzipped = any(f.endswith('.gz') for f in review_files + meta_files)
print(f"\nFiles are gzipped: {is_gzipped}")

Mounted at /content/drive
This will download the raw JSONL files directly



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 38 files:   0%|          | 0/38 [00:00<?, ?it/s]

Download directory: /root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e

Review files found: 0

Metadata files found: 2
  - raw_meta_Electronics (0.0 MB)
  - meta_Electronics.jsonl (5003.1 MB)

Files are gzipped: False


In [ ]:
# ============================================
# INSPECT DOWNLOADED FILES
# ============================================

import glob
import os

download_dir = "/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"

print("All files in download directory:")
print("="*60)

all_files = []
for root, dirs, files in os.walk(download_dir):
    for file in files:
        full_path = os.path.join(root, file)
        size_mb = os.path.getsize(full_path) / 1024 / 1024
        all_files.append((full_path, size_mb))
        print(f"  {size_mb:8.1f} MB  {file}")

print(f"\nTotal files: {len(all_files)}")
print(f"Total size: {sum(s for _, s in all_files):.1f} MB")

# Look specifically for review files
print("\n" + "="*60)
print("LOOKING FOR REVIEW FILES")
print("="*60)

review_files = []
for path, size in all_files:
    if 'review' in path.lower():
        review_files.append(path)
        print(f"  {size:.1f} MB - {os.path.basename(path)}")

if not review_files:
    print("  No files with 'review' in name found!")
    print("\n  All file names:")
    for path, size in all_files:
        print(f"    {os.path.basename(path)}")

All files in download directory:
    2398.7 MB  Electronics.csv
     433.3 MB  Electronics.valid.csv
    1011.5 MB  Electronics.test.csv
     953.9 MB  Electronics.train.csv
     431.8 MB  Electronics.valid.csv
     487.8 MB  Electronics.test.csv
    3241.1 MB  Electronics.train.csv
     218.7 MB  Electronics.valid.csv
     225.4 MB  Electronics.test.csv
    1954.5 MB  Electronics.train.csv
     617.6 MB  Electronics.valid.csv
    1284.5 MB  Electronics.test.csv
    2258.6 MB  Electronics.train.csv
     855.9 MB  Electronics.csv
      90.8 MB  Electronics.valid.csv
      90.8 MB  Electronics.test.csv
     674.4 MB  Electronics.train.csv
     217.0 MB  Electronics.valid.csv
     262.3 MB  Electronics.test.csv
    1721.7 MB  Electronics.train.csv
      64.3 MB  Electronics.valid.csv
      67.2 MB  Electronics.test.csv
     724.4 MB  Electronics.train.csv
     218.7 MB  Electronics.valid.csv
     235.9 MB  Electronics.test.csv
    1746.5 MB  Electronics.train.csv
   21568.5 MB  Electronic

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
from google.colab import drive

drive.mount('/content/drive')

# Paths
DOWNLOAD_DIR = "/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"
REVIEWS_FILE = os.path.join(DOWNLOAD_DIR, "Electronics.jsonl")
META_FILE = os.path.join(DOWNLOAD_DIR, "meta_Electronics.jsonl")
OUTPUT_DIR = '/content/drive/MyDrive/amazon_audio_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✓ Setup complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Setup complete


In [ ]:
AUDIO_KEYWORDS = [
    'headphone', 'headset', 'earphone', 'earbud', 'ear bud', 'earphones',
    'iem', 'in-ear', 'in ear', 'ear monitor',
    'noise canceling', 'noise cancelling',
    'bluetooth ear', 'wireless ear', 'wireless head',
    'over-ear', 'over ear', 'on-ear', 'on ear',
    'audiophile', 'hi-fi', 'hifi',
    'ear tip', 'eartip', 'ear tips',
    'dac', 'amplifier headphone', 'headphone amp',
    'earpad', 'ear pad', 'headphone cable',
    'true wireless', 'tws',
]

AUDIO_BRANDS = [
    'sennheiser', 'beyerdynamic', 'akg', 'audio-technica', 'audiotechnica',
    'sony headphone', 'sony earphone', 'bose', 'airpods',
    'moondrop', 'kz', '7hz', 'truthear', 'tin hifi', 'tinhifi',
    'hifiman', 'audeze', 'focal', 'grado',
    'shure', 'etymotic', 'westone',
    'campfire audio', 'final audio',
    'blon', 'tanchjim', 'dunu', 'fiio',
    'jbl', 'skullcandy', 'beats',
    'galaxy buds', 'pixel buds',
    'nothing ear', 'oneplus buds',
    'anker soundcore', 'soundcore',
    'jabra', 'jaybird',
    'bowers & wilkins', 'bang & olufsen',
    'v-moda', 'meze', 'zmf', 'dan clark', 'stax',
    'thieaudio', 'seeaudio', 'simgot', 'kiwi ears',
    'letshuoer', 'tripowin',
    'sundara', 'hd600', 'hd650', 'hd660', 'hd800',
    'dt770', 'dt880', 'dt990', 'k712', 'k702', 'k371',
    'm50x', 'm40x', 'wh-1000xm',
]

def is_audio(title, categories):
    t = str(title).lower()
    c = str(categories).lower()
    for kw in AUDIO_KEYWORDS:
        if kw in t: return True
    for brand in AUDIO_BRANDS:
        if brand in t: return True
    if 'headphones' in c or 'earbud' in c or 'earphone' in c or 'iem' in c or 'audio' in c:
        return True
    return False

print("✓ Keywords ready")

✓ Keywords ready


In [ ]:
import glob
import os

download_dir = "/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"

# Find exact file names
print("All files in download directory:")
for root, dirs, files in os.walk(download_dir):
    for file in files:
        full_path = os.path.join(root, file)
        size_mb = os.path.getsize(full_path) / 1024 / 1024
        print(f"  {size_mb:8.1f} MB  {file}")

All files in download directory:
    2398.7 MB  Electronics.csv
     433.3 MB  Electronics.valid.csv
    1011.5 MB  Electronics.test.csv
     953.9 MB  Electronics.train.csv
     431.8 MB  Electronics.valid.csv
     487.8 MB  Electronics.test.csv
    3241.1 MB  Electronics.train.csv
     218.7 MB  Electronics.valid.csv
     225.4 MB  Electronics.test.csv
    1954.5 MB  Electronics.train.csv
     617.6 MB  Electronics.valid.csv
    1284.5 MB  Electronics.test.csv
    2258.6 MB  Electronics.train.csv
     855.9 MB  Electronics.csv
      90.8 MB  Electronics.valid.csv
      90.8 MB  Electronics.test.csv
     674.4 MB  Electronics.train.csv
     217.0 MB  Electronics.valid.csv
     262.3 MB  Electronics.test.csv
    1721.7 MB  Electronics.train.csv
      64.3 MB  Electronics.valid.csv
      67.2 MB  Electronics.test.csv
     724.4 MB  Electronics.train.csv
     218.7 MB  Electronics.valid.csv
     235.9 MB  Electronics.test.csv
    1746.5 MB  Electronics.train.csv
   21568.5 MB  Electronic

In [ ]:
# Now use glob to find the exact paths
meta_files = glob.glob(os.path.join(download_dir, "*meta*"))
review_files = glob.glob(os.path.join(download_dir, "*Electronics*"))

print("\nMetadata files found:")
for f in meta_files:
    print(f"  {f}")

print("\nReview files found:")
for f in review_files:
    print(f"  {f}")


Metadata files found:
  /root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw_meta_Electronics

Review files found:
  /root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw_meta_Electronics


In [ ]:
import os

download_dir = "/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"

# Check the raw_meta_Electronics folder
meta_folder = os.path.join(download_dir, "raw_meta_Electronics")
print(f"Contents of raw_meta_Electronics/:")
for item in os.listdir(meta_folder):
    full_path = os.path.join(meta_folder, item)
    if os.path.isdir(full_path):
        print(f"  📁 {item}/")
    else:
        size_mb = os.path.getsize(full_path) / 1024 / 1024
        print(f"  📄 {item} ({size_mb:.1f} MB)")

# Also check what else is in the root
print(f"\nRoot directory contents:")
for item in os.listdir(download_dir):
    full_path = os.path.join(download_dir, item)
    if os.path.isdir(full_path):
        # Check inside folders
        sub_items = os.listdir(full_path)[:5]
        print(f"  📁 {item}/ → {len(sub_items)} files, first: {sub_items}")
    else:
        size_mb = os.path.getsize(full_path) / 1024 / 1024
        print(f"  📄 {item} ({size_mb:.1f} MB)")

Contents of raw_meta_Electronics/:
  📄 full-00002-of-00010.parquet (200.0 MB)
  📄 full-00009-of-00010.parquet (156.4 MB)
  📄 full-00005-of-00010.parquet (184.0 MB)
  📄 full-00000-of-00010.parquet (209.7 MB)
  📄 full-00004-of-00010.parquet (192.9 MB)
  📄 full-00006-of-00010.parquet (182.9 MB)
  📄 full-00001-of-00010.parquet (204.2 MB)
  📄 full-00008-of-00010.parquet (162.4 MB)
  📄 full-00007-of-00010.parquet (174.9 MB)
  📄 full-00003-of-00010.parquet (197.0 MB)

Root directory contents:
  📁 benchmark/ → 2 files, first: ['0core', '5core']
  📁 raw/ → 2 files, first: ['review_categories', 'meta_categories']
  📁 raw_meta_Electronics/ → 5 files, first: ['full-00002-of-00010.parquet', 'full-00009-of-00010.parquet', 'full-00005-of-00010.parquet', 'full-00000-of-00010.parquet', 'full-00004-of-00010.parquet']


In [ ]:
import os

download_dir = "/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"

# Check raw/review_categories and raw/meta_categories
raw_folder = os.path.join(download_dir, "raw")

print("Contents of raw/:")
for item in os.listdir(raw_folder):
    full_path = os.path.join(raw_folder, item)
    if os.path.isdir(full_path):
        sub_items = os.listdir(full_path)[:10]
        print(f"\n  📁 {item}/ → {len(os.listdir(full_path))} files/folders")
        for si in sub_items:
            si_path = os.path.join(full_path, si)
            if os.path.isdir(si_path):
                print(f"      📁 {si}/")
            else:
                size_mb = os.path.getsize(si_path) / 1024 / 1024
                print(f"      📄 {si} ({size_mb:.1f} MB)")


Contents of raw/:

  📁 review_categories/ → 1 files/folders
      📄 Electronics.jsonl (21568.5 MB)

  📁 meta_categories/ → 1 files/folders
      📄 meta_Electronics.jsonl (5003.1 MB)


In [ ]:
import json, os, time
import pandas as pd
import numpy as np
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

DOWNLOAD_DIR = "/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"
REVIEWS_FILE = os.path.join(DOWNLOAD_DIR, "raw", "review_categories", "Electronics.jsonl")
META_FILE = os.path.join(DOWNLOAD_DIR, "raw", "meta_categories", "meta_Electronics.jsonl")
OUTPUT_DIR = '/content/drive/MyDrive/amazon_audio_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✓ Reviews: {os.path.getsize(REVIEWS_FILE)/1024:.0f} GB")
print(f"✓ Metadata: {os.path.getsize(META_FILE)/1024:.0f} GB")
print(f"✓ Output: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Reviews: 22086166 GB
✓ Metadata: 5123188 GB
✓ Output: /content/drive/MyDrive/amazon_audio_data


In [ ]:
AUDIO_KEYWORDS = [
    'headphone', 'headset', 'earphone', 'earbud', 'ear bud', 'earphones',
    'iem', 'in-ear', 'ear monitor', 'noise canceling', 'noise cancelling',
    'bluetooth ear', 'wireless ear', 'wireless head', 'over-ear', 'on-ear',
    'audiophile', 'hi-fi', 'hifi', 'ear tip', 'eartip', 'ear tips',
    'dac', 'headphone amp', 'earpad', 'headphone cable', 'true wireless', 'tws',
]

AUDIO_BRANDS = [
    'sennheiser', 'beyerdynamic', 'akg', 'audio-technica', 'audiotechnica',
    'sony headphone', 'sony earphone', 'bose', 'airpods', 'moondrop', 'kz',
    '7hz', 'truthear', 'tin hifi', 'tinhifi', 'hifiman', 'audeze', 'focal',
    'grado', 'shure', 'etymotic', 'westone', 'campfire audio', 'final audio',
    'blon', 'tanchjim', 'dunu', 'fiio', 'jbl', 'skullcandy', 'beats',
    'galaxy buds', 'pixel buds', 'nothing ear', 'oneplus buds', 'soundcore',
    'jabra', 'jaybird', 'bowers', 'bang & olufsen', 'v-moda', 'meze', 'zmf',
    'dan clark', 'stax', 'thieaudio', 'seeaudio', 'simgot', 'kiwi ears',
    'letshuoer', 'tripowin', 'sundara', 'hd600', 'hd650', 'hd660', 'dt770',
    'dt880', 'dt990', 'k712', 'k702', 'k371', 'm50x', 'm40x', 'wh-1000xm',
]

def is_audio(title, categories):
    t = str(title).lower()
    c = str(categories).lower()
    for kw in AUDIO_KEYWORDS:
        if kw in t: return True
    for brand in AUDIO_BRANDS:
        if brand in t: return True
    for term in ['headphones', 'earbud', 'earphone', 'iem', 'audio']:
        if term in c: return True
    return False

print("✓ Keywords defined")

✓ Keywords defined


In [ ]:
print("Scanning metadata for audio products...")
audio_ids = set()
total = 0

with open(META_FILE, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Products"):
        total += 1
        try:
            item = json.loads(line.strip())
            pid = item.get('parent_asin', '')
            if pid and is_audio(item.get('title', ''), str(item.get('categories', []))):
                audio_ids.add(pid)
        except:
            pass

print(f"Total products: {total:,}")
print(f"Audio products: {len(audio_ids):,}")

with open(f'{OUTPUT_DIR}/audio_ids.json', 'w') as f:
    json.dump(list(audio_ids), f)

print("✓ Saved to Drive")

Scanning metadata for audio products...


Products: 1610012it [01:49, 14652.75it/s]


Total products: 1,610,012
Audio products: 315,586
✓ Saved to Drive


In [ ]:
# Check if audio_ids has anything
print(f"audio_ids count: {len(audio_ids)}")

if len(audio_ids) > 0:
    # Show some sample IDs
    sample_ids = list(audio_ids)[:10]
    print(f"Sample audio IDs: {sample_ids}")

    # Check if reviews file uses parent_asin or asin
    print("\nChecking first 5 reviews to see field names...")
    with open(REVIEWS_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            r = json.loads(line.strip())
            print(f"  Review {i+1}:")
            print(f"    parent_asin: {r.get('parent_asin', 'MISSING')}")
            print(f"    asin: {r.get('asin', 'MISSING')}")
            print(f"    title: {r.get('title', '')[:80]}")
            print()
else:
    print("❌ audio_ids is EMPTY! Metadata scan failed.")
    print("Let's check the metadata file format...")

    # Check first few lines of metadata
    with open(META_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            try:
                r = json.loads(line.strip())
                print(f"\n  Metadata {i+1}:")
                print(f"    Keys: {list(r.keys())}")
                print(f"    parent_asin: {r.get('parent_asin', 'MISSING')}")
                print(f"    title: {str(r.get('title', ''))[:100]}")
                print(f"    categories: {str(r.get('categories', ''))[:100]}")
            except:
                print(f"  Line {i}: {line[:200]}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

OUTPUT_DIR = '/content/drive/MyDrive/amazon_audio_data'

print("Files on Drive:")
if os.path.exists(OUTPUT_DIR):
    for f in os.listdir(OUTPUT_DIR):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
        print(f"  {size:.1f} MB - {f}")
else:
    print("  No files found yet")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files on Drive:
  4.2 MB - audio_ids.json
  3259.1 MB - audio_reviews.jsonl
  767.1 MB - audio_reviews_final.csv


In [ ]:
import pandas as pd
import os

OUTPUT_DIR = '/content/drive/MyDrive/amazon_audio_data'

# Check final CSV
df = pd.read_csv(f'{OUTPUT_DIR}/audio_reviews_final.csv')

print(f"Final dataset:")
print(f"  Reviews: {len(df):,}")
print(f"  Products: {df['parent_asin'].nunique():,}")
print(f"  File size: {os.path.getsize(f'{OUTPUT_DIR}/audio_reviews_final.csv')/1024:.1f} MB")

# Check rating distribution
print(f"\nRating distribution:")
for r in sorted(df['rating'].unique()):
    n = (df['rating'] == r).sum()
    print(f"  {r:.0f}★: {n:,}")

# Check date range
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nSample titles:")
for t in df['title'].dropna().unique()[:10]:
    if len(str(t)) > 5:
        print(f"  - {t[:100]}")

Final dataset:
  Reviews: 2,692,624
  Products: 31,148
  File size: 785471.0 MB

Rating distribution:
  1★: 421,567
  2★: 177,343
  3★: 201,520
  4★: 303,774
  5★: 1,588,420


ValueError: non convertible value 2021-08-02 23:53:58.184 with the unit 'ms', at position 0

In [ ]:
import pandas as pd
import os

OUTPUT_DIR = '/content/drive/MyDrive/amazon_audio_data'
df = pd.read_csv(f'{OUTPUT_DIR}/audio_reviews_final.csv')

# Timestamp is already a datetime string, not Unix ms
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Final dataset:")
print(f"  Reviews: {len(df):,}")
print(f"  Products: {df['parent_asin'].nunique():,}")
print(f"  Users: {df['user_id'].nunique():,}")
print(f"  File size: {os.path.getsize(f'{OUTPUT_DIR}/audio_reviews_final.csv')/1024/1024:.0f} MB")

print(f"\nRating distribution:")
for r in sorted(df['rating'].unique()):
    n = (df['rating'] == r).sum()
    print(f"  {r:.0f}★: {n:,} ({n/len(df)*100:.1f}%)")

print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")

print(f"\nSample products with most reviews:")
top = df['parent_asin'].value_counts().head(10)
for pid, count in top.items():
    title = df[df['parent_asin'] == pid]['title'].iloc[0]
    print(f"  {count:,} reviews - {str(title)[:80]}")

Final dataset:
  Reviews: 2,692,624
  Products: 31,148


KeyError: 'user_id'

In [ ]:
import pandas as pd
import os

OUTPUT_DIR = '/content/drive/MyDrive/amazon_audio_data'
df = pd.read_csv(f'{OUTPUT_DIR}/audio_reviews_final.csv')

# Timestamp is already a datetime string
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Final dataset:")
print(f"  Reviews: {len(df):,}")
print(f"  Products: {df['parent_asin'].nunique():,}")
print(f"  File size: {os.path.getsize(f'{OUTPUT_DIR}/audio_reviews_final.csv')/1024/1024:.0f} MB")

print(f"\nRating distribution:")
for r in sorted(df['rating'].unique()):
    n = (df['rating'] == r).sum()
    print(f"  {r:.0f}★: {n:,} ({n/len(df)*100:.1f}%)")

print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")

print(f"\nSample products with most reviews:")
top = df['parent_asin'].value_counts().head(10)
for pid, count in top.items():
    title = df[df['parent_asin'] == pid]['title'].iloc[0]
    print(f"  {count:,} reviews - {str(title)[:80]}")

Final dataset:
  Reviews: 2,692,624
  Products: 31,148
  File size: 767 MB

Rating distribution:
  1★: 421,567 (15.7%)
  2★: 177,343 (6.6%)
  3★: 201,520 (7.5%)
  4★: 303,774 (11.3%)
  5★: 1,588,420 (59.0%)

Date range: 2019-01-01 00:00:10.870000 to 2023-09-12 19:20:18.911000

Sample products with most reviews:
  500 reviews - Great for the wondering mind
  500 reviews - Works well for the money
  500 reviews - Deep Sleeper’s MUST own this AMAZING alarm clock
  500 reviews - Very easy to use
  500 reviews - Splitter
  500 reviews - it is a useless piece of crap
  500 reviews - Love rubber bumper
  500 reviews - JUNK
  500 reviews - Great
  500 reviews - the best of my setup
